# Run Holosoma Sandbox Many Data

本 notebook 是 sandbox retargeting 的命令行控制面。它发现可用数据、生成任务表、按需调用 sandbox entry script，并检查结果 `.npz` 与 Hessian sidecar 是否存在。

当前 preset 将实验语义拆开：regular 跑原生 OMOMO 和 climb；extra 只跑 OMOMO 的真实 thumb/pinky keypoints；climb 的 spherehand extra 使用单独的 offset 变体。Retarget/LAFAN 转换优先使用同名非空原始 BVH，并在没有可用 BVH 时回退到经过验证的 `.npz`；不符合单位、上轴、帧率、脚尖和手性合同的输入会显示在 rejected 表中，不会进入 retargeting 任务。


## 1 User Settings

Normal runs should only edit the next code cell.


In [8]:
# Edit only this cell for normal runs.

EXPERIMENT = "all_pairs"  # "holosoma_regular", "holosoma_extra", "holosoma_climb_extra_offsets", "lafan_regular", "lafan_removed", or "all_pairs"
TASK_FILTER: list[str] = []     # [] means all tasks selected by EXPERIMENT

RUN_RETARGETING = True
FORCE_RETARGETING = False
MAX_TASKS_TO_RUN: int | None = None


## 2 Setup And Derived Configuration


In [9]:
from __future__ import annotations

import json
import os
import subprocess
import sys
from dataclasses import asdict, dataclass
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display


def find_workspace_root(start: Path | None = None) -> Path:
    path = (start or Path.cwd()).resolve()
    for candidate in [path, *path.parents]:
        if (candidate / "Retarget").is_dir() and (candidate / "holosoma_retargeting").is_dir():
            return candidate
    raise RuntimeError("Cannot find Retarget-whole workspace root")


EXPERIMENT_PRESETS = {
    "holosoma_regular": {"dataset_group": "holosoma_native", "sandbox_variant": "regular"},
    "holosoma_extra": {"dataset_group": "holosoma_omomo", "sandbox_variant": "extra"},
    "holosoma_climb_extra_offsets": {"dataset_group": "holosoma_climb", "sandbox_variant": "extra_offsets"},
    "lafan_regular": {"dataset_group": "lafan", "sandbox_variant": "regular"},
    "lafan_removed": {"dataset_group": "lafan", "sandbox_variant": "removed"},
}
EXPERIMENT_GROUPS = {
    "all_pairs": [
        "holosoma_regular",
        "holosoma_extra",
        "holosoma_climb_extra_offsets",
        "lafan_regular",
        "lafan_removed",
    ],
}
SANDBOX_VARIANTS = {
    "regular": {
        "variant_name": "sandbox_experimental",
        "entry_script": "examples/robot_retarget_sandbox_experimental.py",
    },
    "extra": {
        "variant_name": "sandbox_extra_keypoints_experimental",
        "entry_script": "examples/robot_retarget_sandbox_extra_keypoints_experimental.py",
    },
    "extra_offsets": {
        "variant_name": "sandbox_extra_keypoints_offsets_experimental",
        "entry_script": "examples/robot_retarget_sandbox_extra_keypoints_offsets_experimental.py",
    },
    "removed": {
        "variant_name": "sandbox_removed_experimental",
        "entry_script": "examples/robot_retarget_sandbox_removed_experimental.py",
    },
}
EXPERIMENT_CHOICES = sorted([*EXPERIMENT_PRESETS, *EXPERIMENT_GROUPS])
if EXPERIMENT in EXPERIMENT_GROUPS:
    ACTIVE_EXPERIMENT_NAMES = list(EXPERIMENT_GROUPS[EXPERIMENT])
elif EXPERIMENT in EXPERIMENT_PRESETS:
    ACTIVE_EXPERIMENT_NAMES = [EXPERIMENT]
else:
    raise ValueError(f"Unknown EXPERIMENT={EXPERIMENT!r}; expected one of {EXPERIMENT_CHOICES}")

ACTIVE_EXPERIMENT_CONFIGS = []
for experiment_name in ACTIVE_EXPERIMENT_NAMES:
    preset = EXPERIMENT_PRESETS[experiment_name]
    sandbox_variant = preset["sandbox_variant"]
    variant_info = SANDBOX_VARIANTS[sandbox_variant]
    ACTIVE_EXPERIMENT_CONFIGS.append(
        {
            "experiment": experiment_name,
            "dataset_group": preset["dataset_group"],
            "sandbox_variant": sandbox_variant,
            "variant_name": variant_info["variant_name"],
            "entry_script": variant_info["entry_script"],
        }
    )

CONDA_ENV = "robot"
ROBOT = "g1"
EXTRA_ARGS: list[str] = []

WORKSPACE_ROOT = find_workspace_root()
RETARGET_DIR = WORKSPACE_ROOT / "Retarget"
RETARGET_NPZ_DIR = RETARGET_DIR / "bvh_npz"
RETARGET_BVH_DIR = RETARGET_NPZ_DIR / "lafan1"
HOLOSOMA_REPO = WORKSPACE_ROOT / "holosoma_retargeting"
HOLOSOMA_ROOT = HOLOSOMA_REPO / "holosoma_retargeting"

if str(HOLOSOMA_REPO) not in sys.path:
    sys.path.insert(0, str(HOLOSOMA_REPO))

from holosoma_retargeting.config_types.data_type import LAFAN_DEMO_JOINTS
from holosoma_retargeting.data_conversion.convert_lafan_bvh_to_holosoma import (
    prepare_lafan_bvh_for_holosoma,
)

EXTRA_KEYPOINTS_VARIANT_NAMES = {
    SANDBOX_VARIANTS["extra"]["variant_name"],
    SANDBOX_VARIANTS["extra_offsets"]["variant_name"],
}
EXTRA_OFFSETS_VARIANT_NAME = SANDBOX_VARIANTS["extra_offsets"]["variant_name"]

GENERATED_DATA_ROOT = HOLOSOMA_ROOT / "generated_data"
RETARGET_LAFAN_DIR = GENERATED_DATA_ROOT / "retarget_lafan"
EXPERIMENT_RESULTS_ROOT = HOLOSOMA_ROOT / "experiment_results" / "exploratory" / "laplacian_hessian"
RESULTS_BASE = EXPERIMENT_RESULTS_ROOT


def results_root_for(variant_name: str) -> Path:
    return RESULTS_BASE / variant_name / ROBOT


RESULTS_ROOTS = sorted({results_root_for(config["variant_name"]) for config in ACTIVE_EXPERIMENT_CONFIGS})
LOG_ROOT = EXPERIMENT_RESULTS_ROOT / "logs" / "run_sandbox_many_data"

RUN_PREPARE_RETARGET_INPUTS = RUN_RETARGETING
FORCE_PREPARE_RETARGET_INPUTS = FORCE_RETARGETING

# The raw climb folders contain g1_29dof_spherehand_w_multi_boxes.xml.
CLIMB_EXTRA_ARGS = ["--robot-config.robot-urdf-file", "models/g1/g1_29dof_spherehand.urdf"]
LAFAN_HESSIAN_EXTRA_ARGS = [
    "--task-config.ground-range",
    "-10",
    "10",
    "--retargeter.hessian-record-frame-stride",
    "10",
    "--retargeter.hessian-record-inner-stride",
    "1",
]
EXTRA_SMPLH_HAND_KEYS = ["L_Thumb3", "L_Pinky3", "R_Thumb3", "R_Pinky3"]
EXTRA_MOCAP_HAND_KEYS = ["LeftHandThumb3", "LeftHandPinky3", "RightHandThumb3", "RightHandPinky3"]

REQUIRED_HESSIAN_FIELDS = [
    "hessian_component_names",
    "hessian_component_matrices",
    "hessian_frame",
    "hessian_inner_iter",
    "hessian_q_active",
    "hessian_q_eval",
    "hessian_lap_J_V",
    "hessian_lap_J_lap",
    "hessian_lap_weights",
    "hessian_lap_vertices",
    "hessian_lap_num_robot_vertices",
    "hessian_lap_robot_link_keys",
]

print("workspace:", WORKSPACE_ROOT)
print("Holosoma root:", HOLOSOMA_ROOT)
print("generated data root:", GENERATED_DATA_ROOT)
print("experiment results root:", EXPERIMENT_RESULTS_ROOT)
print("experiment:", EXPERIMENT)
print("expanded experiments:", ACTIVE_EXPERIMENT_NAMES)
for config in ACTIVE_EXPERIMENT_CONFIGS:
    print(
        "  -",
        config["experiment"],
        "dataset=",
        config["dataset_group"],
        "variant=",
        config["sandbox_variant"],
        "->",
        config["variant_name"],
    )
print("results roots:")
for root in RESULTS_ROOTS:
    print("  -", root)


workspace: /home/cipher/Codes/Academic-Projects/Retarget-whole
Holosoma root: /home/cipher/Codes/Academic-Projects/Retarget-whole/holosoma_retargeting/holosoma_retargeting
generated data root: /home/cipher/Codes/Academic-Projects/Retarget-whole/holosoma_retargeting/holosoma_retargeting/generated_data
experiment results root: /home/cipher/Codes/Academic-Projects/Retarget-whole/holosoma_retargeting/holosoma_retargeting/experiment_results/exploratory/laplacian_hessian
experiment: all_pairs
expanded experiments: ['holosoma_regular', 'holosoma_extra', 'holosoma_climb_extra_offsets', 'lafan_regular', 'lafan_removed']
  - holosoma_regular dataset= holosoma_native variant= regular -> sandbox_experimental
  - holosoma_extra dataset= holosoma_omomo variant= extra -> sandbox_extra_keypoints_experimental
  - holosoma_climb_extra_offsets dataset= holosoma_climb variant= extra_offsets -> sandbox_extra_keypoints_offsets_experimental
  - lafan_regular dataset= lafan variant= regular -> sandbox_experim

## 3 Helpers


In [10]:
@dataclass(frozen=True)
class SandboxTask:
    stable_id: str
    source_kind: str
    task_type: str
    data_format: str
    task_name: str
    data_path: Path
    save_dir: Path
    output_path: Path
    input_path: Path | None
    variant_name: str
    entry_script: str
    source_frames: int | None
    expected_loaded_frames: int | None
    prepare_status: str
    extra_args: tuple[str, ...] = ()


def output_path_for(task_type: str, save_dir: Path, task_name: str) -> Path:
    if task_type == "robot_only":
        return save_dir / f"{task_name}.npz"
    return save_dir / f"{task_name}_original.npz"


def task_selected(task: SandboxTask) -> bool:
    if not TASK_FILTER:
        return True
    text = " ".join([task.stable_id, task.source_kind, task.task_type, task.data_format, task.task_name])
    return any(token in text for token in TASK_FILTER)


def sidecar_path_for(result_path: Path) -> Path | None:
    if not result_path.exists():
        return None
    data = np.load(result_path, allow_pickle=True)
    if "hessian_components_file" not in data.files:
        return None
    sidecar_path = result_path.with_name(str(data["hessian_components_file"].item()))
    if not sidecar_path.exists():
        return None
    return sidecar_path


def expected_extra_keypoint_keys(task: SandboxTask) -> list[str]:
    if task.variant_name not in EXTRA_KEYPOINTS_VARIANT_NAMES:
        return []
    if task.data_format == "smplh":
        return EXTRA_SMPLH_HAND_KEYS
    if (
        task.variant_name == EXTRA_OFFSETS_VARIANT_NAME
        and task.data_format == "mocap"
        and "g1_29dof_spherehand.urdf" in " ".join(task.extra_args)
    ):
        return EXTRA_MOCAP_HAND_KEYS
    return []


def expected_hessian_record_config(task: SandboxTask) -> dict[str, bool | int]:
    if task.data_format == "lafan":
        return {
            "hessian_record_enabled": True,
            "hessian_record_frame_stride": 10,
            "hessian_record_inner_stride": 1,
        }
    return {}


def sidecar_status_for(task: SandboxTask) -> tuple[Path | None, str]:
    sidecar_path = sidecar_path_for(task.output_path)
    if sidecar_path is None:
        return None, "missing"
    if (
        task.input_path is not None
        and task.input_path.exists()
        and task.output_path.stat().st_mtime_ns < task.input_path.stat().st_mtime_ns
    ):
        return sidecar_path, "stale_input_newer"

    sidecar_data = None
    expected_keys = expected_extra_keypoint_keys(task)
    if expected_keys:
        sidecar_data = np.load(sidecar_path, allow_pickle=True)
        if "hessian_lap_robot_link_keys" not in sidecar_data.files:
            return sidecar_path, "stale_missing_hessian_lap_robot_link_keys"
        saved_keys = {str(key) for key in sidecar_data["hessian_lap_robot_link_keys"]}
        missing_keys = [key for key in expected_keys if key not in saved_keys]
        if missing_keys:
            return sidecar_path, "stale_missing_extra_keys:" + ",".join(missing_keys)

    expected_record_config = expected_hessian_record_config(task)
    if expected_record_config:
        if sidecar_data is None:
            sidecar_data = np.load(sidecar_path, allow_pickle=True)
        for key, expected_value in expected_record_config.items():
            if key not in sidecar_data.files:
                return sidecar_path, "stale_missing_" + key
            actual_value = sidecar_data[key].item()
            if actual_value != expected_value:
                return sidecar_path, f"stale_{key}:{actual_value}"

    if task.variant_name == EXTRA_OFFSETS_VARIANT_NAME and task.data_format == "mocap":
        if sidecar_data is None:
            sidecar_data = np.load(sidecar_path, allow_pickle=True)
        if "hessian_lap_robot_point_offsets" not in sidecar_data.files:
            return sidecar_path, "stale_missing_hessian_lap_robot_point_offsets"
        offsets = np.asarray(sidecar_data["hessian_lap_robot_point_offsets"], dtype=float)
        if offsets.size == 0 or not np.any(np.linalg.norm(offsets.reshape(-1, 3), axis=1) > 0):
            return sidecar_path, "stale_zero_extra_offsets"

    return sidecar_path, "ready"


def cli_path(path: Path | str) -> str:
    path = Path(path)
    try:
        return str(path.resolve().relative_to(HOLOSOMA_ROOT))
    except ValueError:
        return str(path)


def command_for(task: SandboxTask) -> list[str]:
    return [
        "conda",
        "run",
        "--no-capture-output",
        "-n",
        CONDA_ENV,
        "python",
        "-u",
        task.entry_script,
        "--task-type",
        task.task_type,
        "--robot",
        ROBOT,
        "--data-format",
        task.data_format,
        "--task-name",
        task.task_name,
        "--data-path",
        cli_path(task.data_path),
        "--save-dir",
        cli_path(task.save_dir),
        *task.extra_args,
        *EXTRA_ARGS,
    ]


def command_text(cmd: list[str]) -> str:
    return " ".join(str(part) for part in cmd)


def run_command(cmd: list[str], log_path: Path, cwd: Path) -> int:
    log_path.parent.mkdir(parents=True, exist_ok=True)
    print("$", command_text(cmd))
    env = os.environ.copy()
    env["PYTHONUNBUFFERED"] = "1"
    with log_path.open("w", encoding="utf-8") as log_file:
        process = subprocess.Popen(
            cmd,
            cwd=str(cwd),
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            bufsize=1,
            env=env,
        )
        assert process.stdout is not None
        for line in process.stdout:
            print(line, end="", flush=True)
            log_file.write(line)
            log_file.flush()
        return process.wait()


## 4 Retarget LAFAN Conversion

每个 Retarget LAFAN task 优先从同名非空原始 BVH 做前向运动学；没有可用 BVH 时才读取 `.npz`。两条路径都必须通过 cm、Y-up、30 FPS、脚尖有效性和坐标手性检查。生成的 `.npy` 保持 Y-up，Holosoma loader 再用 proper rotation 转成 Z-up；首帧 Hips 的水平位置归零。Holosoma 原生 OMOMO 和 climb 数据不在本节处理。

In [11]:
RETARGET_LAFAN_CONVERSION_VERSION = 2
RETARGET_LAFAN_EXPECTED_FPS = 30.0
RETARGET_LAFAN_CM_TO_M = 0.01

JOINT_NAME_ALIASES = {
    "LeftToe": "LeftToeBase",
    "RightToe": "RightToeBase",
}


@dataclass(frozen=True)
class RetargetLafanPreparation:
    task_name: str
    source_path: Path
    source_format: str
    output_path: Path
    source_frames: int
    expected_loaded_frames: int
    status: str
    valid: bool
    diagnostics: dict[str, float | str]


def normalize_source_joint_names(joint_names: tuple[str, ...]) -> list[str]:
    return [JOINT_NAME_ALIASES.get(name, name) for name in joint_names]


def inspect_retarget_lafan_source(
    data: np.lib.npyio.NpzFile,
) -> tuple[np.ndarray, list[str], dict[str, float | str], list[str]]:
    positions = np.asarray(data["positions"], dtype=np.float64)
    source_joint_names = normalize_source_joint_names(tuple(str(x) for x in data["joint_names"]))
    issues: list[str] = []
    diagnostics: dict[str, float | str] = {}

    if positions.ndim != 3 or positions.shape[-1] != 3:
        return positions, source_joint_names, diagnostics, [f"positions shape must be (T, J, 3), got {positions.shape}"]
    if not np.all(np.isfinite(positions)):
        issues.append("positions contain non-finite values")

    missing = [name for name in LAFAN_DEMO_JOINTS if name not in source_joint_names]
    if missing:
        issues.append("missing joints: " + ",".join(missing))
        return positions, source_joint_names, diagnostics, issues

    frame_time = float(np.asarray(data["frame_time"]).item()) if "frame_time" in data.files else np.nan
    source_fps = 1.0 / frame_time if np.isfinite(frame_time) and frame_time > 0 else np.nan
    diagnostics["source_fps"] = source_fps
    if not np.isfinite(source_fps) or not np.isclose(source_fps, RETARGET_LAFAN_EXPECTED_FPS, atol=0.05):
        issues.append(f"expected 30 FPS, got {source_fps:.6g}")

    idx = {name: source_joint_names.index(name) for name in LAFAN_DEMO_JOINTS}
    hips_to_head = positions[:, idx["Head"]] - positions[:, idx["Hips"]]
    hip_head_length = float(np.median(np.linalg.norm(hips_to_head, axis=1)))
    up_axis = "xyz"[int(np.argmax(np.median(np.abs(hips_to_head), axis=0)))]
    diagnostics["median_hips_head_source_units"] = hip_head_length
    diagnostics["detected_up_axis"] = up_axis
    if not 40.0 <= hip_head_length <= 100.0:
        issues.append(f"expected centimeter-scale skeleton, median Hips-Head={hip_head_length:.6g}")
    if up_axis != "y":
        issues.append(f"expected Y-up source, detected {up_axis.upper()}-up")

    toe_vectors = []
    toe_zero_fractions = []
    for side in ("Left", "Right"):
        toe = positions[:, idx[f"{side}ToeBase"]]
        foot = positions[:, idx[f"{side}Foot"]]
        toe_vectors.append(toe - foot)
        toe_zero_fractions.append(float(np.mean(np.linalg.norm(toe, axis=1) < 1e-8)))
    median_toe_length = float(np.median(np.linalg.norm(np.concatenate(toe_vectors, axis=0), axis=1)))
    max_toe_zero_fraction = max(toe_zero_fractions)
    diagnostics["median_foot_toe_source_units"] = median_toe_length
    diagnostics["max_toe_zero_fraction"] = max_toe_zero_fraction
    if not 2.0 <= median_toe_length <= 35.0:
        issues.append(f"invalid foot-toe length, median={median_toe_length:.6g}")
    if max_toe_zero_fraction > 0.95:
        issues.append(f"toe positions are zero placeholders, fraction={max_toe_zero_fraction:.3f}")

    forward = 0.5 * (toe_vectors[0] + toe_vectors[1])
    left = positions[:, idx["LeftUpLeg"]] - positions[:, idx["RightUpLeg"]]
    up = hips_to_head
    denominator = np.linalg.norm(forward, axis=1) * np.linalg.norm(left, axis=1) * np.linalg.norm(up, axis=1)
    valid = denominator > 1e-8
    if np.any(valid):
        handedness = np.einsum("ij,ij->i", np.cross(forward[valid], left[valid]), up[valid]) / denominator[valid]
        median_handedness = float(np.median(handedness))
    else:
        median_handedness = np.nan
    diagnostics["median_forward_left_up_handedness"] = median_handedness
    if not np.isfinite(median_handedness) or median_handedness <= 0.2:
        issues.append(f"invalid forward/left/up handedness={median_handedness:.6g}")

    return positions, source_joint_names, diagnostics, issues


def prepare_retarget_lafan_npz(npz_path: Path) -> RetargetLafanPreparation:
    data = np.load(npz_path, allow_pickle=True)
    positions_cm, source_joint_names, diagnostics, issues = inspect_retarget_lafan_source(data)
    task_name = npz_path.stem
    output_path = RETARGET_LAFAN_DIR / f"{task_name}.npy"
    source_frames = int(positions_cm.shape[0])
    if issues:
        return RetargetLafanPreparation(
            task_name=task_name,
            source_path=npz_path,
            source_format="validated_npz",
            output_path=output_path,
            source_frames=source_frames,
            expected_loaded_frames=source_frames,
            status="invalid: " + " | ".join(issues),
            valid=False,
            diagnostics=diagnostics,
        )

    order = [source_joint_names.index(name) for name in LAFAN_DEMO_JOINTS]
    positions_m = positions_cm[:, order, :] * RETARGET_LAFAN_CM_TO_M
    hips_idx = LAFAN_DEMO_JOINTS.index("Hips")
    positions_m[:, :, 0] -= positions_m[0, hips_idx, 0]
    positions_m[:, :, 2] -= positions_m[0, hips_idx, 2]

    metadata_path = output_path.with_suffix(".conversion.json")
    source_stat = npz_path.stat()
    expected_metadata = {
        "conversion_version": RETARGET_LAFAN_CONVERSION_VERSION,
        "source_path": str(npz_path.resolve()),
        "source_size": source_stat.st_size,
        "source_mtime_ns": source_stat.st_mtime_ns,
        "source_frames": source_frames,
        "source_fps": diagnostics["source_fps"],
        "source_coordinates": "centimeters_y_up",
        "generated_coordinates": "meters_y_up_first_frame_hips_xz_centered",
    }
    try:
        saved_metadata = json.loads(metadata_path.read_text(encoding="utf-8"))
    except (FileNotFoundError, json.JSONDecodeError):
        saved_metadata = None
    prepared_is_current = output_path.exists() and saved_metadata == expected_metadata
    needs_write = FORCE_PREPARE_RETARGET_INPUTS or not prepared_is_current

    output_path.parent.mkdir(parents=True, exist_ok=True)
    if RUN_PREPARE_RETARGET_INPUTS and needs_write:
        np.save(output_path, positions_m.astype(np.float32))
        metadata_path.write_text(json.dumps(expected_metadata, indent=2) + "\n", encoding="utf-8")
        status = "written"
    elif prepared_is_current:
        status = "current"
    else:
        status = "stale_or_missing_not_written"

    return RetargetLafanPreparation(
        task_name=task_name,
        source_path=npz_path,
        source_format="validated_npz",
        output_path=output_path,
        source_frames=source_frames,
        expected_loaded_frames=int(positions_m.shape[0]),
        status=status,
        valid=True,
        diagnostics=diagnostics,
    )


def prepare_retarget_lafan_bvh(bvh_path: Path) -> RetargetLafanPreparation:
    task_name = bvh_path.stem
    output_path = RETARGET_LAFAN_DIR / f"{task_name}.npy"
    try:
        result = prepare_lafan_bvh_for_holosoma(
            bvh_path,
            output_path,
            force=FORCE_PREPARE_RETARGET_INPUTS,
            write=RUN_PREPARE_RETARGET_INPUTS,
        )
    except ValueError as exc:
        return RetargetLafanPreparation(
            task_name=task_name,
            source_path=bvh_path,
            source_format="raw_bvh",
            output_path=output_path,
            source_frames=0,
            expected_loaded_frames=0,
            status=f"invalid: {exc}",
            valid=False,
            diagnostics={},
        )
    return RetargetLafanPreparation(
        task_name=task_name,
        source_path=bvh_path,
        source_format="raw_bvh",
        output_path=output_path,
        source_frames=result.source_frames,
        expected_loaded_frames=result.output_frames,
        status=result.status,
        valid=True,
        diagnostics=result.diagnostics,
    )

## 5 Build Task Table

OMOMO 和 climb 使用原始 Holosoma 数据路径；Retarget LAFAN 使用上一步由 raw BVH 或已验证 `.npz` 生成的 `lafan` `.npy`。

In [12]:
tasks: list[SandboxTask] = []
retarget_lafan_preparation_cache: dict[Path, RetargetLafanPreparation] = {}
invalid_lafan_rows: list[dict[str, object]] = []
VALID_DATASET_GROUPS = {"holosoma_native", "holosoma_omomo", "holosoma_climb", "lafan"}

for experiment_config in ACTIVE_EXPERIMENT_CONFIGS:
    dataset_group = experiment_config["dataset_group"]
    sandbox_variant = experiment_config["sandbox_variant"]
    variant_name = experiment_config["variant_name"]
    entry_script = experiment_config["entry_script"]
    variant_suffix = "" if sandbox_variant == "regular" else f"_{sandbox_variant}"

    if dataset_group not in VALID_DATASET_GROUPS:
        raise ValueError(f"Unsupported dataset_group={dataset_group!r}")

    if dataset_group in {"holosoma_native", "holosoma_omomo"}:
        omomo_dir = HOLOSOMA_ROOT / "demo_data" / "OMOMO_new"
        save_dir = results_root_for(variant_name) / "object_interaction" / "omomo"
        stable_prefix = f"omomo{variant_suffix}"
        source_kind = f"holosoma_omomo{variant_suffix}"
        for pt_path in sorted(omomo_dir.glob("*.pt")):
            task_name = pt_path.stem
            tasks.append(
                SandboxTask(
                    stable_id=f"{stable_prefix}:{task_name}",
                    source_kind=source_kind,
                    task_type="object_interaction",
                    data_format="smplh",
                    task_name=task_name,
                    data_path=omomo_dir,
                    save_dir=save_dir,
                    output_path=output_path_for("object_interaction", save_dir, task_name),
                    input_path=pt_path,
                    variant_name=variant_name,
                    entry_script=entry_script,
                    source_frames=None,
                    expected_loaded_frames=None,
                    prepare_status="native",
                )
            )

    if dataset_group in {"holosoma_native", "holosoma_climb"}:
        climb_dir = HOLOSOMA_ROOT / "demo_data" / "climb"
        save_dir = results_root_for(variant_name) / "climbing" / "mocap_climb"
        stable_prefix = f"climb{variant_suffix}"
        source_kind = f"holosoma_climb_native{variant_suffix}"
        for task_dir in sorted(path for path in climb_dir.iterdir() if path.is_dir()):
            npy_files = sorted(task_dir.glob("*.npy"))
            if not npy_files:
                continue
            source_npy = npy_files[0]
            source_frames = int(np.load(source_npy, mmap_mode="r").shape[0])
            task_name = task_dir.name
            tasks.append(
                SandboxTask(
                    stable_id=f"{stable_prefix}:{task_name}",
                    source_kind=source_kind,
                    task_type="climbing",
                    data_format="mocap",
                    task_name=task_name,
                    data_path=climb_dir,
                    save_dir=save_dir,
                    output_path=output_path_for("climbing", save_dir, task_name),
                    input_path=source_npy,
                    variant_name=variant_name,
                    entry_script=entry_script,
                    source_frames=source_frames,
                    expected_loaded_frames=int(np.ceil(source_frames / 4.0)),
                    prepare_status="native",
                    extra_args=tuple(CLIMB_EXTRA_ARGS),
                )
            )

    if dataset_group == "lafan":
        save_dir = results_root_for(variant_name) / "robot_only" / "retarget_lafan"
        stable_prefix = f"retarget_lafan{variant_suffix}"
        source_stems = sorted(
            {path.stem for path in RETARGET_NPZ_DIR.glob("*.npz")}
            | {path.stem for path in RETARGET_BVH_DIR.glob("*.bvh") if path.stat().st_size > 0}
        )
        for task_name in source_stems:
            npz_path = RETARGET_NPZ_DIR / f"{task_name}.npz"
            bvh_path = RETARGET_BVH_DIR / f"{task_name}.bvh"
            source_path = bvh_path if bvh_path.is_file() and bvh_path.stat().st_size > 0 else npz_path
            if source_path not in retarget_lafan_preparation_cache:
                if source_path.suffix.lower() == ".bvh":
                    preparation = prepare_retarget_lafan_bvh(source_path)
                else:
                    preparation = prepare_retarget_lafan_npz(source_path)
                retarget_lafan_preparation_cache[source_path] = preparation
            preparation = retarget_lafan_preparation_cache[source_path]
            if not preparation.valid:
                invalid_lafan_rows.append(
                    {
                        "source_path": cli_path(preparation.source_path),
                        "source_format": preparation.source_format,
                        "task_name": preparation.task_name,
                        "status": preparation.status,
                        **preparation.diagnostics,
                    }
                )
                continue
            tasks.append(
                SandboxTask(
                    stable_id=f"{stable_prefix}:{preparation.task_name}",
                    source_kind=f"retarget_lafan_{preparation.source_format}_converted{variant_suffix}",
                    task_type="robot_only",
                    data_format="lafan",
                    task_name=preparation.task_name,
                    data_path=RETARGET_LAFAN_DIR,
                    save_dir=save_dir,
                    output_path=output_path_for("robot_only", save_dir, preparation.task_name),
                    input_path=preparation.output_path,
                    variant_name=variant_name,
                    entry_script=entry_script,
                    source_frames=preparation.source_frames,
                    expected_loaded_frames=preparation.expected_loaded_frames,
                    prepare_status=preparation.status,
                    extra_args=tuple(LAFAN_HESSIAN_EXTRA_ARGS),
                )
            )

selected_tasks = [task for task in tasks if task_selected(task)]
invalid_lafan_df = pd.DataFrame(invalid_lafan_rows)
if not invalid_lafan_df.empty:
    invalid_lafan_df = invalid_lafan_df.drop_duplicates(subset=["source_path", "status"])

task_df = pd.DataFrame(
    [
        {
            **asdict(task),
            "data_path": cli_path(task.data_path),
            "save_dir": cli_path(task.save_dir),
            "output_path": cli_path(task.output_path),
            "input_path": cli_path(task.input_path) if task.input_path is not None else None,
            "variant_name": task.variant_name,
            "entry_script": task.entry_script,
            "command": command_text(command_for(task)),
            "output_exists": task.output_path.exists(),
            "sidecar_exists": sidecar_path is not None,
            "sidecar_ready": sidecar_status == "ready",
            "sidecar_status": sidecar_status,
        }
        for task in selected_tasks
        for sidecar_path, sidecar_status in [sidecar_status_for(task)]
    ]
)

print(f"experiment: {EXPERIMENT}")
print(f"expanded experiments: {ACTIVE_EXPERIMENT_NAMES}")
print(f"all tasks: {len(tasks)}")
print(f"selected tasks: {len(selected_tasks)}")
if task_df.empty:
    print("No runnable tasks matched the current settings.")
else:
    display(task_df[[
        "stable_id",
        "source_kind",
        "task_type",
        "data_format",
        "task_name",
        "variant_name",
        "entry_script",
        "source_frames",
        "expected_loaded_frames",
        "prepare_status",
        "output_exists",
        "sidecar_exists",
        "sidecar_ready",
        "sidecar_status",
    ]])
if not invalid_lafan_df.empty:
    print("Rejected Retarget/LAFAN inputs:")
    display(invalid_lafan_df)


experiment: all_pairs
expanded experiments: ['holosoma_regular', 'holosoma_extra', 'holosoma_climb_extra_offsets', 'lafan_regular', 'lafan_removed']
all tasks: 20
selected tasks: 20


,stable_id,source_kind,task_type,data_format,task_name,variant_name,entry_script,source_frames,expected_loaded_frames,prepare_status,output_exists,sidecar_exists,sidecar_ready,sidecar_status
0,omomo:sub10_largebox_049,holosoma_omomo,object_interaction,smplh,sub10_largebox_049,sandbox_experimental,examples/robot_retarget_sandbox_experimental.py,NaN,NaN,native,True,True,True,ready
1,omomo:sub3_largebox_003,holosoma_omomo,object_interaction,smplh,sub3_largebox_003,sandbox_experimental,examples/robot_retarget_sandbox_experimental.py,NaN,NaN,native,True,True,True,ready
2,climb:mocap_climb_seq_0,holosoma_climb_native,climbing,mocap,mocap_climb_seq_0,sandbox_experimental,examples/robot_retarget_sandbox_experimental.py,2801.0,701.0,native,True,True,True,ready
3,climb:mocap_climb_seq_1,holosoma_climb_native,climbing,mocap,mocap_climb_seq_1,sandbox_experimental,examples/robot_retarget_sandbox_experimental.py,2423.0,606.0,native,True,True,True,ready
4,climb:mocap_climb_seq_2,holosoma_climb_native,climbing,mocap,mocap_climb_seq_2,sandbox_experimental,examples/robot_retarget_sandbox_experimental.py,3011.0,753.0,native,True,True,True,ready
5,climb:mocap_climb_seq_3,holosoma_climb_native,climbing,mocap,mocap_climb_seq_3,sandbox_experimental,examples/robot_retarget_sandbox_experimental.py,2401.0,601.0,native,True,True,True,ready
6,climb:mocap_climb_seq_4,holosoma_climb_native,climbing,mocap,mocap_climb_seq_4,sandbox_experimental,examples/robot_retarget_sandbox_experimental.py,2303.0,576.0,native,True,True,True,ready
7,omomo_extra:sub10_largebox_049,holosoma_omomo_extra,object_interaction,smplh,sub10_largebox_049,sandbox_extra_keypoints_experimental,examples/robot_retarget_sandbox_extra_keypoint...,NaN,NaN,native,True,True,True,ready
8,omomo_extra:sub3_largebox_003,holosoma_omomo_extra,object_interaction,smplh,sub3_largebox_003,sandbox_extra_keypoints_experimental,examples/robot_retarget_sandbox_extra_keypoint...,NaN,NaN,native,True,True,True,ready
9,climb_extra_offsets:mocap_climb_seq_0,holosoma_climb_native_extra_offsets,climbing,mocap,mocap_climb_seq_0,sandbox_extra_keypoints_offsets_experimental,examples/robot_retarget_sandbox_extra_keypoint...,2801.0,701.0,native,True,True,True,ready


## 6 Run Sandbox Tasks

本节直接执行任务表里的命令。`RUN_RETARGETING=False` 时只打印将要执行的命令。

In [13]:
tasks_to_run = selected_tasks
if MAX_TASKS_TO_RUN is not None:
    tasks_to_run = tasks_to_run[: int(MAX_TASKS_TO_RUN)]

run_records = []
for task in tasks_to_run:
    cmd = command_for(task)
    log_path = LOG_ROOT / task.variant_name / f"{task.stable_id.replace(':', '__').replace('/', '_')}.log"
    sidecar_path, sidecar_status = sidecar_status_for(task)
    if sidecar_status == "ready" and not FORCE_RETARGETING:
        status = "exists_with_ready_sidecar"
        returncode = 0
        print(f"skip existing result with ready Hessian sidecar: {task.output_path}")
    elif RUN_RETARGETING:
        if sidecar_path is not None and sidecar_status != "ready":
            print(f"rerun stale Hessian sidecar ({sidecar_status}): {task.output_path}")
        returncode = run_command(cmd, log_path=log_path, cwd=HOLOSOMA_ROOT)
        status = "ok" if returncode == 0 else "failed"
    else:
        status = "pending"
        returncode = None
        print("not run:", command_text(cmd))

    run_records.append(
        {
            "stable_id": task.stable_id,
            "task_name": task.task_name,
            "output_path": task.output_path,
            "log_path": log_path,
            "status": status,
            "sidecar_status_before": sidecar_status,
            "returncode": returncode,
        }
    )

run_df = pd.DataFrame(
    [
        {
            **record,
            "output_path": cli_path(record["output_path"]),
            "log_path": cli_path(record["log_path"]),
        }
        for record in run_records
    ]
)
display(run_df)

skip existing result with ready Hessian sidecar: /home/cipher/Codes/Academic-Projects/Retarget-whole/holosoma_retargeting/holosoma_retargeting/experiment_results/exploratory/laplacian_hessian/sandbox_experimental/g1/object_interaction/omomo/sub10_largebox_049_original.npz
skip existing result with ready Hessian sidecar: /home/cipher/Codes/Academic-Projects/Retarget-whole/holosoma_retargeting/holosoma_retargeting/experiment_results/exploratory/laplacian_hessian/sandbox_experimental/g1/object_interaction/omomo/sub3_largebox_003_original.npz
skip existing result with ready Hessian sidecar: /home/cipher/Codes/Academic-Projects/Retarget-whole/holosoma_retargeting/holosoma_retargeting/experiment_results/exploratory/laplacian_hessian/sandbox_experimental/g1/climbing/mocap_climb/mocap_climb_seq_0_original.npz
skip existing result with ready Hessian sidecar: /home/cipher/Codes/Academic-Projects/Retarget-whole/holosoma_retargeting/holosoma_retargeting/experiment_results/exploratory/laplacian_hes

,stable_id,task_name,output_path,log_path,status,sidecar_status_before,returncode
0,omomo:sub10_largebox_049,sub10_largebox_049,experiment_results/exploratory/laplacian_hessi...,experiment_results/exploratory/laplacian_hessi...,exists_with_ready_sidecar,ready,0
1,omomo:sub3_largebox_003,sub3_largebox_003,experiment_results/exploratory/laplacian_hessi...,experiment_results/exploratory/laplacian_hessi...,exists_with_ready_sidecar,ready,0
2,climb:mocap_climb_seq_0,mocap_climb_seq_0,experiment_results/exploratory/laplacian_hessi...,experiment_results/exploratory/laplacian_hessi...,exists_with_ready_sidecar,ready,0
3,climb:mocap_climb_seq_1,mocap_climb_seq_1,experiment_results/exploratory/laplacian_hessi...,experiment_results/exploratory/laplacian_hessi...,exists_with_ready_sidecar,ready,0
4,climb:mocap_climb_seq_2,mocap_climb_seq_2,experiment_results/exploratory/laplacian_hessi...,experiment_results/exploratory/laplacian_hessi...,exists_with_ready_sidecar,ready,0
5,climb:mocap_climb_seq_3,mocap_climb_seq_3,experiment_results/exploratory/laplacian_hessi...,experiment_results/exploratory/laplacian_hessi...,exists_with_ready_sidecar,ready,0
6,climb:mocap_climb_seq_4,mocap_climb_seq_4,experiment_results/exploratory/laplacian_hessi...,experiment_results/exploratory/laplacian_hessi...,exists_with_ready_sidecar,ready,0
7,omomo_extra:sub10_largebox_049,sub10_largebox_049,experiment_results/exploratory/laplacian_hessi...,experiment_results/exploratory/laplacian_hessi...,exists_with_ready_sidecar,ready,0
8,omomo_extra:sub3_largebox_003,sub3_largebox_003,experiment_results/exploratory/laplacian_hessi...,experiment_results/exploratory/laplacian_hessi...,exists_with_ready_sidecar,ready,0
9,climb_extra_offsets:mocap_climb_seq_0,mocap_climb_seq_0,experiment_results/exploratory/laplacian_hessi...,experiment_results/exploratory/laplacian_hessi...,exists_with_ready_sidecar,ready,0


## 7 Check Result And Hessian Sidecars

本节扫描所有 selected tasks，不只扫描本次运行的任务。后续打开 `multi_laplacian_hessian_spectrum_diagnosis.ipynb` 即可批量分析已有 sidecar。

In [14]:
check_rows = []
for task in selected_tasks:
    sidecar_path, sidecar_status = sidecar_status_for(task)
    row = {
        "stable_id": task.stable_id,
        "task_name": task.task_name,
        "result_exists": task.output_path.exists(),
        "sidecar_exists": sidecar_path is not None,
        "sidecar_ready": sidecar_status == "ready",
        "sidecar_status": sidecar_status,
        "result_path": cli_path(task.output_path),
        "sidecar_path": cli_path(sidecar_path) if sidecar_path is not None else None,
        "qpos_shape": None,
        "missing_hessian_fields": None,
    }
    if task.output_path.exists():
        result_data = np.load(task.output_path, allow_pickle=True)
        if "qpos" in result_data.files:
            row["qpos_shape"] = tuple(result_data["qpos"].shape)
    if sidecar_path is not None:
        hessian_data = np.load(sidecar_path, allow_pickle=True)
        missing = [name for name in REQUIRED_HESSIAN_FIELDS if name not in hessian_data.files]
        row["missing_hessian_fields"] = missing
    check_rows.append(row)

check_df = pd.DataFrame(check_rows)
if check_df.empty:
    print("No selected tasks to check.")
    completed_df = check_df.copy()
else:
    display(check_df[[
        "stable_id",
        "result_exists",
        "sidecar_exists",
        "sidecar_ready",
        "sidecar_status",
        "qpos_shape",
        "missing_hessian_fields",
    ]])
    completed_df = check_df[check_df["sidecar_ready"]].copy()
print(f"completed results with ready Hessian sidecars: {len(completed_df)}")
if len(completed_df):
    display(completed_df[["stable_id", "result_path", "sidecar_path"]])

,stable_id,result_exists,sidecar_exists,sidecar_ready,sidecar_status,qpos_shape,missing_hessian_fields
0,omomo:sub10_largebox_049,True,True,True,ready,"(222, 43)",[]
1,omomo:sub3_largebox_003,True,True,True,ready,"(196, 43)",[]
2,climb:mocap_climb_seq_0,True,True,True,ready,"(701, 36)",[]
3,climb:mocap_climb_seq_1,True,True,True,ready,"(606, 36)",[]
4,climb:mocap_climb_seq_2,True,True,True,ready,"(753, 36)",[]
5,climb:mocap_climb_seq_3,True,True,True,ready,"(601, 36)",[]
6,climb:mocap_climb_seq_4,True,True,True,ready,"(576, 36)",[]
7,omomo_extra:sub10_largebox_049,True,True,True,ready,"(222, 43)",[]
8,omomo_extra:sub3_largebox_003,True,True,True,ready,"(196, 43)",[]
9,climb_extra_offsets:mocap_climb_seq_0,True,True,True,ready,"(701, 36)",[]


completed results with ready Hessian sidecars: 20


,stable_id,result_path,sidecar_path
0,omomo:sub10_largebox_049,experiment_results/exploratory/laplacian_hessi...,experiment_results/exploratory/laplacian_hessi...
1,omomo:sub3_largebox_003,experiment_results/exploratory/laplacian_hessi...,experiment_results/exploratory/laplacian_hessi...
2,climb:mocap_climb_seq_0,experiment_results/exploratory/laplacian_hessi...,experiment_results/exploratory/laplacian_hessi...
3,climb:mocap_climb_seq_1,experiment_results/exploratory/laplacian_hessi...,experiment_results/exploratory/laplacian_hessi...
4,climb:mocap_climb_seq_2,experiment_results/exploratory/laplacian_hessi...,experiment_results/exploratory/laplacian_hessi...
5,climb:mocap_climb_seq_3,experiment_results/exploratory/laplacian_hessi...,experiment_results/exploratory/laplacian_hessi...
6,climb:mocap_climb_seq_4,experiment_results/exploratory/laplacian_hessi...,experiment_results/exploratory/laplacian_hessi...
7,omomo_extra:sub10_largebox_049,experiment_results/exploratory/laplacian_hessi...,experiment_results/exploratory/laplacian_hessi...
8,omomo_extra:sub3_largebox_003,experiment_results/exploratory/laplacian_hessi...,experiment_results/exploratory/laplacian_hessi...
9,climb_extra_offsets:mocap_climb_seq_0,experiment_results/exploratory/laplacian_hessi...,experiment_results/exploratory/laplacian_hessi...


## 8 Next Steps

- For normal use, edit only **User Settings** at the top.
- `EXPERIMENT` selects a valid data/algorithm pair:
  - `holosoma_regular`: native Holosoma OMOMO + climb with regular sandbox.
  - `holosoma_extra`: native Holosoma OMOMO only with true thumb/pinky extra keypoints.
  - `holosoma_climb_extra_offsets`: native Holosoma climb only with spherehand local-offset thumb/pinky keypoints.
  - `lafan_regular`: Retarget/LAFAN with regular sandbox.
  - `lafan_removed`: Retarget/LAFAN with removed-variable sandbox.
  - `all_pairs`: all valid pairs above.
- `TASK_FILTER` can select one sequence, for example `["dance1"]` or `["mocap_climb_seq_0"]`.
- LAFAN conversion prefers a same-name nonempty raw BVH and falls back to a validated cm/Y-up/30 FPS `.npz`. It centers the first-frame Hips horizontal position and uses full sequences. `aiming1_subject1` is rebuilt from its 7184-frame raw BVH, so the invalid 60 FPS NPZ and its zero toe placeholders are not used.
- Invalid Retarget/LAFAN inputs are listed in `invalid_lafan_df` and excluded from the task table. Conversion is controlled by `RUN_RETARGETING` / `FORCE_RETARGETING`.
- LAFAN sandbox commands use the fixed ground range `[-10, 10]` and record Hessian diagnostics every 10 frames and every inner iteration; other tasks use full Hessian diagnostic recording.
- Subprocess output is streamed to the notebook cell and also written to the per-task log file.
- Existing result `.npz` files with ready Hessian sidecars are skipped when `FORCE_RETARGETING=False`; a result older than its converted input is marked `stale_input_newer` and rerun.
- After sidecars exist, run `multi_laplacian_hessian_spectrum_diagnosis.ipynb` and select a result with `RESULT_INDEX` or `RESULT_PATH`.
